## Dask Futures for general-purpose Python

This notebook shows how to use **Dask Futures** to run ordinary Python functions in parallel on a cluster.

No xarray / Dask Array — just normal functions, `submit` / `map`, and `gather`.

**When to use Futures:** custom Python (loops, file IO, non-Dask libraries) that you want to run concurrently on workers.

In [ ]:
import time
from dask.distributed import Client, LocalCluster, as_completed, wait

### 1. Start a local cluster

In [ ]:
cluster = LocalCluster(n_workers=4, threads_per_worker=1)
client = Client(cluster)

print(client.dashboard_link)
client

### 2. A plain Python function

This is ordinary code — nothing Dask-specific. Imagine each call is a slow file parse, API request, or tile of custom processing.

In [ ]:
def process_item(item: int, scale: float = 1.0) -> dict:
    """
    Stand-in for any custom Python work.
    Sleeps briefly to simulate real cost, then returns a small result.
    """
    time.sleep(0.5)
    value = (item ** 2) * scale
    return {"item": item, "value": value}

### 3. Run one task with `client.submit`

`submit` schedules the function on the cluster and returns a **Future** immediately (it does not wait for the result).

In [ ]:
future = client.submit(process_item, 7, scale=2.0)
display(future)

In [ ]:
# Block until this future finishes and fetch the result
future.result()
display(future)

### 4. Run many tasks in parallel

Two equivalent styles:

- **`client.map`** — apply one function to a sequence
- **`client.submit` in a list comprehension** — more flexible when arguments differ per task

In [ ]:
items = list(range(12))

# Style A: map
futures = client.map(process_item, items, scale=1.5)

# Style B (equivalent idea):
# futures = [client.submit(process_item, i, scale=1.5) for i in items]

futures[:3]

### 5. Wait for results

- **`gather`** — wait and return the values on the client
- **`wait`** — wait until done, but do not fetch results
- **`as_completed`** — yield futures as each one finishes

In [ ]:
%%time
results = client.gather(futures)
results

In [ ]:
# wait: barrier only (no results)
futures2 = client.map(process_item, range(4))
wait(futures2)
print("All done:", all(f.done() for f in futures2))

# Still need gather/result if you want the values
client.gather(futures2)

In [ ]:
# as_completed: handle each result as soon as it arrives
futures3 = client.map(process_item, range(6))

for fut in as_completed(futures3):
    print(fut.result())

### 6. Share large inputs with `scatter`

If many tasks need the **same** large object, scatter it once so each `submit` passes a Future reference instead of re-pickling the data every time.

Use `broadcast=True` when every worker needs a copy.

In [ ]:
def score_against_lookup(item: int, lookup: dict) -> float:
    time.sleep(0.2)
    return item * lookup.get(item % 5, 1)

# Pretend this is a large shared table
lookup = {i: i * 10 for i in range(5)}

lookup_future = client.scatter(lookup, broadcast=True)

futures = [client.submit(score_against_lookup, i, lookup_future) for i in range(8)]
client.gather(futures)

### 7. Chain futures (task dependencies)

Pass a Future as an argument to another `submit`. The downstream task waits for the upstream result automatically.

In [ ]:
def double(x: float) -> float:
    time.sleep(0.2)
    return x * 2


def add(a: float, b: float) -> float:
    time.sleep(0.2)
    return a + b


a = client.submit(process_item, 3)           # -> dict
a_value = client.submit(lambda d: d["value"], a)
b = client.submit(double, a_value)
c = client.submit(add, a_value, b)           # depends on both

c.result()

### 8. Batching (optional)

For very large job lists, submit in batches to avoid flooding the scheduler.

In [ ]:
def chunks(lst, n=5):
    return [lst[i:i + n] for i in range(0, len(lst), n)]


all_items = list(range(20))
all_results = []
print(f"All items: {all_items}")

for batch in chunks(all_items, 5):
    print(f"Batch: {batch}")
    batch_futures = client.map(process_item, batch, scale=2)
    all_results.extend(client.gather(batch_futures))

len(all_results), all_results[:3]

### 9. Clean up

In [ ]:
client.close()
cluster.close()

### Takeaways

1. Futures parallelise **ordinary Python functions**, not Dask Array / xarray graphs.
2. `submit` / `map` return Futures immediately; use `gather`, `result`, `wait`, or `as_completed` to synchronise.
3. `scatter` (optionally `broadcast=True`) avoids re-shipping shared inputs.
4. Pass Futures into later `submit` calls to build dependent workflows.
5. For pure array math, prefer Dask Array / xarray instead of wrapping everything in Futures.